In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Data Reading

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@azuredatabricksete.dfs.core.windows.net/customers")

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data")

In [0]:
df_domain_name = df.withColumn("domain_name", split(col("email"),"@")[1])

In [0]:
df_domain_name.display()

### Aggregating Customers basis Domain Name

In [0]:
df_aggregation = df_domain_name.groupBy("domain_name").agg(count("customer_id").alias("total_customers")).sort("total_customers",asc=False)
df_aggregation.display()

In [0]:
df_gmail = df_domain_name.filter(col("domain_name").like("%gmail.com%"))

In [0]:
df_final = df_domain_name.withColumn("full_name", concat(col("first_name"),lit(" "),col("last_name"))).drop("first_name","last_name")

In [0]:
df_final.display()

# Data Writing

In [0]:
df_final.write.format("delta").mode("append").save("abfss://silver@azuredatabricksete.dfs.core.windows.net/customers")

In [0]:
df_confirmation=spark.read.format("delta").load("abfss://silver@azuredatabricksete.dfs.core.windows.net/customers")
df_confirmation.display()

In [0]:
%sql
CREATE  TABLE IF NOT EXISTS catalog_databricks_ete.silver.customers_silver
USING DELTA
LOCATION 'abfss://silver@azuredatabricksete.dfs.core.windows.net/customers'